# 10 — SOM Training

Train a Self-Organising Map on the preprocessed feature stack.
Saves the trained codebook and FAISS index for use in `11_som_mapping.ipynb`.

**Input**: precomputed training data from `06_combine_features.ipynb`,
feature selection from `07_feature_selection.ipynb`

**Output**: `final_som_codebook.pkl`, `som_production.index`

### Load precomputed training data

In [ ]:
import config
import gc
import json
import pickle
import numpy as np
import xarray as xr
import s3fs
from s3_utils import load_zarr

s3 = s3fs.S3FileSystem(anon=False)
prep = config.S3_PROCESSED + '/som_prep'

# Training matrix (already scaled + post-processed)
s3.get(f'{prep}/X_train_scaled.npy', '/tmp/X_train_scaled.npy')
X_train_scaled = np.load('/tmp/X_train_scaled.npy')

# Valid indices for full-grid assignment
s3.get(f'{prep}/valid_indices.npy', '/tmp/valid_indices.npy')
valid_indices = np.load('/tmp/valid_indices.npy')
n_valid = len(valid_indices)

# Scaler (for transforming chunks during assignment)
s3.get(f'{prep}/scaler.pkl', '/tmp/scaler.pkl')
with open('/tmp/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Feature names
s3.get(f'{prep}/feature_names.json', '/tmp/feature_names.json')
with open('/tmp/feature_names.json') as f:
    feature_names = json.load(f)
n_features = len(feature_names)

# Grid metadata
s3.get(f'{prep}/grid_meta.json', '/tmp/grid_meta.json')
with open('/tmp/grid_meta.json') as f:
    grid_meta = json.load(f)
grid_shape = tuple(grid_meta['grid_shape'])
n_pixels = grid_meta['n_pixels']

print(f'Training matrix: {X_train_scaled.shape}')
print(f'Valid pixels: {n_valid:,} / {n_pixels:,}')
print(f'Features ({n_features}): {feature_names}')


### Apply feature selection

In [ ]:
# Load selected features from notebook 07
with open('selected_features.json') as f:
    selected_features = json.load(f)

# Get the original full feature list (before selection)
all_feature_names = feature_names.copy()

print(f'Selected {len(selected_features)} / {len(feature_names)} features:')
print(selected_features)

# Subset training matrix
sel_idx = [all_feature_names.index(f) for f in selected_features]
X_train_scaled = X_train_scaled[:, sel_idx]

# Subset the scaler to match (only keep selected feature columns)
scaler.center_ = scaler.center_[sel_idx]
scaler.scale_ = scaler.scale_[sel_idx]

feature_names = selected_features
n_features = len(feature_names)

print(f'Training matrix after selection: {X_train_scaled.shape}')


### Post-scale function

In [ ]:
def post_scale(X):
    """Domain-specific adjustments after RobustScaler."""
    X = X.copy()
    if 'forest_edge_dist' in feature_names:
        X[:, feature_names.index('forest_edge_dist')] = np.clip(
            X[:, feature_names.index('forest_edge_dist')], -5, 5)
    for name in feature_names:
        if name.startswith('lc_'):
            X[:, feature_names.index(name)] *= 3.0
    return np.clip(X, -5, 5)


### Convergence analysis (200 epochs, staged)

Run this cell to evaluate QE/TE across training stages.
Used to determine the optimal epoch count. **Skip for production runs.**

In [ ]:
import time
import os
import faiss
import somoclu
import matplotlib.pyplot as plt

som_side = 64

# Train/val split (80/20) for honest QE/TE evaluation
n_train = X_train_scaled.shape[0]
n_val = int(n_train * 0.2)
rng = np.random.default_rng(42)
val_idx = rng.choice(n_train, n_val, replace=False)
train_idx = np.setdiff1d(np.arange(n_train), val_idx)

X_fit = X_train_scaled[train_idx].astype(np.float32)
X_val = X_train_scaled[val_idx].astype(np.float32)
print(f'Train: {len(X_fit):,}, Val: {len(X_val):,}')

som = somoclu.Somoclu(
    n_columns=som_side, n_rows=som_side,
    gridtype='hexagonal', maptype='toroid', initialization='pca',
)

STAGES = [20, 40, 60, 80, 100, 150, 200]
prev_epochs = 0
history = []

plot_features = feature_names[:12]
n_plot = len(plot_features)
n_cols = 4
n_rows_plot = (n_plot + n_cols - 1) // n_cols

os.makedirs('som_plots', exist_ok=True)

for target_epoch in STAGES:
    epochs_this_stage = target_epoch - prev_epochs
    frac_start = prev_epochs / STAGES[-1]
    frac_end = target_epoch / STAGES[-1]

    r0 = max(som_side * (1 - frac_start), 1)
    rN = max(som_side * (1 - frac_end), 1)
    s0 = 0.05 * (1 - frac_start) + 0.01 * frac_start
    sN = 0.05 * (1 - frac_end) + 0.01 * frac_end

    t0 = time.time()
    som.train(
        data=X_fit, epochs=epochs_this_stage,
        radius0=r0, radiusN=rN, scale0=s0, scaleN=sN,
    )
    dt = time.time() - t0

    # Evaluate on HELD-OUT validation set
    cb = som.codebook.reshape(-1, n_features).astype('float32')
    index = faiss.IndexFlatL2(n_features)
    index.add(cb)
    D, I = index.search(X_val, k=2)

    qe = np.mean(np.sqrt(D[:, 0]))
    bmu1, bmu2 = I[:, 0], I[:, 1]
    dr = np.minimum(np.abs(bmu1 // som_side - bmu2 // som_side),
                    som_side - np.abs(bmu1 // som_side - bmu2 // som_side))
    dc = np.minimum(np.abs(bmu1 % som_side - bmu2 % som_side),
                    som_side - np.abs(bmu1 % som_side - bmu2 % som_side))
    te = 1.0 - ((dr <= 1) & (dc <= 1)).mean()
    n_used = len(np.unique(I[:, 0]))
    history.append((target_epoch, qe, te, n_used))

    print(f'Epoch {target_epoch:3d}  val_QE={qe:.4f}  val_TE={te:.4f}  '
          f'BMUs={n_used}/{som_side**2}  ({dt:.1f}s)')

    # Save codebook feature maps to disk
    codebook = som.codebook.reshape(som_side, som_side, n_features)
    fig, axes = plt.subplots(n_rows_plot, n_cols, figsize=(4*n_cols, 3.5*n_rows_plot))
    for fi, feat in enumerate(plot_features):
        ax = axes.flat[fi]
        im = ax.imshow(codebook[:, :, feature_names.index(feat)], cmap='viridis')
        ax.set_title(feat, fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046)
        ax.set_xticks([]); ax.set_yticks([])
    for fi in range(n_plot, n_rows_plot * n_cols):
        axes.flat[fi].set_visible(False)
    fig.suptitle(f'Epoch {target_epoch} — val_QE={qe:.4f}  val_TE={te:.4f}', fontsize=12)
    plt.tight_layout()
    fig.savefig(f'som_plots/codebook_epoch_{target_epoch:03d}.png', dpi=100, bbox_inches='tight')
    plt.close(fig)

    prev_epochs = target_epoch

print(f'\nFinal: val_QE={history[-1][1]:.4f}, val_TE={history[-1][2]:.4f}')
print(f'Plots saved to som_plots/')


### Production training (150 epochs)

Based on convergence analysis, 150 epochs is the sweet spot.
Saves codebook + FAISS index to disk.

In [ ]:
import time
import numpy as np
import faiss
import somoclu
import pickle

# --- 1. Configuration & Hyperparameters ---
SOM_SIDE = 64
N_FEATURES = X_train_scaled.shape[1]
TOTAL_EPOCHS = 150

# Parameters derived from our "Epoch 150" sweet spot analysis
R_START, R_END = SOM_SIDE, 16
S_START, S_END = 0.05, 0.016

# --- 2. Train/Val Split (80/20) ---
n_samples = X_train_scaled.shape[0]
n_val = int(n_samples * 0.2)
rng = np.random.default_rng(42)  # Fixed seed for reproducibility
val_idx = rng.choice(n_samples, n_val, replace=False)
train_idx = np.setdiff1d(np.arange(n_samples), val_idx)

X_fit = X_train_scaled[train_idx].astype(np.float32)
X_val = X_train_scaled[val_idx].astype(np.float32)

print(f"Starting Training: {len(X_fit):,} samples")
print(f"Validation Set:    {len(X_val):,} samples")

# --- 3. Initialize & Train ---
som = somoclu.Somoclu(
    n_columns=SOM_SIDE, n_rows=SOM_SIDE,
    gridtype='hexagonal', 
    maptype='toroid', 
    initialization='pca'
)

t0 = time.time()
som.train(
    data=X_fit, 
    epochs=TOTAL_EPOCHS,
    radius0=R_START, radiusN=R_END, 
    scale0=S_START, scaleN=S_END,
)
duration = time.time() - t0
print(f"Training complete in {duration:.1f}s")

# --- 4. Final Metric Recalculation (on Val Set) ---
# Extract and flatten codebook: (64, 64, 21) -> (4096, 21)
cb_final = som.codebook.reshape(-1, N_FEATURES).astype('float32')

# Build FAISS Index for exact L2 search
index = faiss.IndexFlatL2(N_FEATURES)
index.add(cb_final)

# Search 2 nearest neighbors for validation samples
D, I = index.search(X_val, k=2)

# 1. Quantization Error (Mean L2 distance to 1st BMU)
qe = np.mean(np.sqrt(D[:, 0]))

# 2. Topographic Error (Check if 1st and 2nd BMUs are neighbors on toroid grid)
bmu1, bmu2 = I[:, 0], I[:, 1]
bmu1_r, bmu1_c = bmu1 // SOM_SIDE, bmu1 % SOM_SIDE
bmu2_r, bmu2_c = bmu2 // SOM_SIDE, bmu2 % SOM_SIDE

# Calculate distances with toroid wrap-around
dr = np.minimum(np.abs(bmu1_r - bmu2_r), SOM_SIDE - np.abs(bmu1_r - bmu2_r))
dc = np.minimum(np.abs(bmu1_c - bmu2_c), SOM_SIDE - np.abs(bmu1_c - bmu2_c))
te = 1.0 - ((dr <= 1) & (dc <= 1)).mean()

# 3. Neuron Utilization
n_used = len(np.unique(bmu1))

print("\n--- Final Production Metrics ---")
print(f"Val QE: {qe:.4f}")
print(f"Val TE: {te:.4f}")
print(f"BMU Usage: {n_used}/{SOM_SIDE**2} ({n_used/SOM_SIDE**2:.1%})")

# --- 5. Export for Mapping Phase ---
# We save the flattened codebook and the FAISS index for the 160M pixel pass
with open('final_som_codebook.pkl', 'wb') as f:
    pickle.dump(cb_final, f)
faiss.write_index(index, "som_production.index")

print("\nModel saved. Ready for the 160M pixel assignment pass.")

### Inspect codebook

In [ ]:
import matplotlib.pyplot as plt

codebook = som.codebook.reshape(som_side, som_side, n_features)

fig, axes = plt.subplots(3, 4, figsize=(18, 18))
for i, feat in enumerate(selected_features):
    ax = axes[i // 4, i % 4]
    im = ax.imshow(codebook[:, :, feature_names.index(feat)], cmap='viridis')
    ax.set_title(f'SOM: {feat}')
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()
